# 1. Segmentation: judgments to sentences

Argument mining for Indian criminal defamation judgments, split into eight notebooks. Run them in order the first time. After that, each notebook reloads what it needs from `argument_mining/`, so you can re-run any one of them on its own.

| Notebook | What it does | Reads | Writes |
|---|---|---|---|
| `01_segmentation` | Splits judgments into paragraphs and sentences | `full_df_clean2.csv` | `intermediate/sentences.pkl` |
| `02_gold_standard` | Aligns lawyer highlights to sentences, holds out test judgments, lawyer agreement | sentences, `gold_annotations.csv` | `intermediate/gold_sentences.csv`, `test_judgments.csv` |
| `03_keyword_methods` | M0, M1, M1n, M1p | sentences, gold | `predictions/M0…`, `M1…`, `intermediate/m1_hits.csv` |
| `04_exemplar_similarity` | InLegalBERT embeddings, M2 | sentences, gold | embedding cache, `predictions/M2…` |
| `05_finetuned_inlegalbert` | M6s (reported), M6g, M6: InLegalBERT fine-tuned on M1 labels. Stage 2 did not beat stage 1, so there is no M6t | sentences, gold, M1 predictions | `predictions/M6…`, stage 1 model, `stage2_grid.csv/.png` |
| `06_method_comparison` | M4, scores with confidence intervals, error review | all predictions | `method_comparison.csv/.png`, error sheet |
| `07_corpus_outcomes` | Best method on all judgments, outcome tests | sentences, predictions, comparison | `case_level_categories.csv` |

**Dev and test.** `02_gold_standard` holds out at least three annotated judgments as a test set. Methods learn and tune only on the other (dev) judgments, the best method is chosen on dev, and the test scores in `06_method_comparison` are the ones to report. Do not change a method because of a test result; if you do, the test score is no longer an honest estimate.

Shared settings, category definitions and helpers are in `argmine_common.py`. Predictions are stored by sentence position, so if you re-run 01 or 02, re-run the method notebooks (03 to 05) as well.

## Overview

These notebooks replace classifies sentences of judicial reasoning into the five lawyer defined categories and evaluates every method against the lawyers' gold annotations.

| Code | Category |
|---|---|
| C1 | Ingredients of the offence and exceptions |
| C2 | Balance of speech and reputation, public officials |
| C3 | Procedural grounds |
| C4 | Insufficient evidence |
| C5 | Abuse of process |

**Methods compared** (all evaluated on the same sentences against the same gold labels):

| ID | Method | Role in the paper |
|---|---|---|
| M0 | Keyword list from lawyers | Baseline |
| M1 | Revised phrase lists, precedent citations, scope blockers | Method 1 |
| M1n | M1, also discarding negated hits ("not an abuse of process") | Negation ablation |
| M1p | M1, skipping sentences that restate a party's submission | Speaker ablation |
| M2 | InLegalBERT similarity to lawyer highlighted exemplars | Method 2 |
| M4 | M1 or M2 (hybrid) | Combined |
| M6s | InLegalBERT fine-tuned on M1 labels only. M6 adds the lawyer highlights and M6g uses highlights alone; neither beat M6s | Step 3: learns from steps 1 and 2 |

**Sections used:** `court_reasoning` and `conclusion`. Highlights that still cannot be placed are attached as single rescued sentences from the full judgment, not by classifying every section. A four-section ablation (`law_analysis` + `precedent_analysis`) is saved in `argument_mining/archive_four_sections/`.

**Pipeline:** segment those sections into numbered paragraphs and sentences, align gold highlights to sentences, run each method, evaluate with leave one judgment out where a method learns from gold data, then apply the best method to all 297 judgments.

**Changes from `nidhi_working.ipynb`**
1. Segmentation now splits on the judgment's own paragraph numbers, then sentences. The old blank line split produced one "paragraph" per judgment.
2. Evaluation is against lawyer annotations, not against keyword labels, so the scores measure correctness rather than agreement with the keyword rules.
3. The normalized percentage analysis is removed: five independent yes/no probabilities are not shares of a judge's reasoning.

## Setup

In [1]:
from argmine_common import *

## Load the corpus

In [2]:
corpus = load_corpus()
TITLE = titles(corpus)
print(f"{len(corpus)} judgments, {(corpus.reason_text.str.len() > 0).sum()} with reasoning text")

298 judgments, 297 with reasoning text


## Segmentation: numbered paragraphs, then sentences

Indian judgments number their paragraphs ("13. In the present case..."). We split on the longest increasing run of paragraph numbers, which skips paragraph numbers that appear inside quoted precedents. Judgments without numbering fall back to blank lines, then to the whole text. Each paragraph is then split into sentences, protecting legal abbreviations (Cr.P.C., vs., No., S. 482, initials).

Each sentence also gets two flags:
* `in_quote`: inside quoted statute or precedent text
* `party_argument`: the court restating a party's submission rather than giving its own reasoning

In [3]:
PAGE_NOISE = [
    r"Indian Kanoon\s*-\s*http\S+(\s+\d+)?",
    r"https?://\S+",
    r"\b\d{1,3}\s+of\s+\d{1,3}\s+\d{1,3}\s+[A-Z][A-Za-z.]*(?:-[A-Za-z.()]+)*-\d+-\d{4}(\s*\(O&M\))?",
]

HEADER_LINE = re.compile(r"^.{0,120}\bvs?\.?\s.{0,100}\bon \d{1,2} \w+,? \d{4}\s*$", re.M)  # PDF page header

def clean_text(text):
    text = HEADER_LINE.sub(" ", str(text))
    for pat in PAGE_NOISE:
        text = re.sub(pat, " ", text)
    text = re.sub(r"(\w)-\n\s*(\w)", r"\1\2", text)        # words broken across lines
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    return text.strip()


PARA_START = re.compile(r'(?:^|(?<=[\s"”.:;]))\(?(\d{1,3})[.)]\s+(?=["“(\[]?[A-Z])')

def split_numbered_paragraphs(text, max_gap=3, min_paras=3):
    # Longest chain of paragraph numbers, each 1 to max_gap above the previous one.
    cands = [(m.start(), int(m.group(1))) for m in PARA_START.finditer(text)]
    n = len(cands)
    if n >= min_paras:
        best, prev = [1] * n, [-1] * n
        for i in range(n):
            for j in range(i):
                if cands[j][1] < cands[i][1] <= cands[j][1] + max_gap and best[j] + 1 > best[i]:
                    best[i], prev[i] = best[j] + 1, j
        end = int(np.argmax(best))
        if best[end] >= min_paras:
            chain = []
            while end != -1:
                chain.append(cands[end]); end = prev[end]
            chain.reverse()
            bounds = [p for p, _ in chain] + [len(text)]
            paras = []
            head = text[:bounds[0]].strip()
            if len(head.split()) >= 5:
                paras.append((None, head))
            for i, (_, num) in enumerate(chain):
                body = re.sub(r"^\s*\(?\d{1,3}[.)]\s+", "", text[bounds[i]:bounds[i + 1]]).strip()
                paras.append((num, body))
            return paras, "numbered"
    blocks = [b.strip() for b in re.split(r"\n\s*\n+", text) if b.strip()]
    if len(blocks) > 1:
        return [(None, b) for b in blocks], "blank_line"
    return [(None, text.strip())], "none"


ABBREVIATIONS = ["vs", "v", "no", "nos", "sec", "secs", "s", "ss", "art", "arts", "cl", "cr", "crl",
                 "p", "pp", "para", "paras", "pg", "mr", "mrs", "ms", "dr", "smt", "shri", "sri", "kum",
                 "ltd", "pvt", "co", "corp", "inc", "ors", "anr", "viz", "ld", "supp", "vol", "ed",
                 "ex", "exh", "rs", "govt", "dept", "jt", "addl", "spl", "asst", "dist", "distt", "hon"]
DOT = "․"   # placeholder for protected periods
_ABBR = re.compile(r"\b(" + "|".join(ABBREVIATIONS) + r")\.(?=\s|\d|\(|$)", re.I)
_DOTTED = re.compile(r"\b(?:[A-Za-z]{1,4}\.){2,}")
_INITIAL = re.compile(r"\b([A-Z])\.(?=\s*[A-Za-z(])")
_DECIMAL = re.compile(r"(?<=\d)\.(?=\d)")
_SPLIT = re.compile(r'(?:(?<=[.!?])|(?<=[.!?]["”’)\]]))\s+(?=["“‘(\[]?[A-Z0-9])')

def split_sentences(text, min_words=4, max_words=120):
    t = re.sub(r"\s+", " ", text).strip()
    t = _DOTTED.sub(lambda m: m.group(0).replace(".", DOT), t)
    t = _ABBR.sub(lambda m: m.group(1) + DOT, t)
    t = _INITIAL.sub(lambda m: m.group(1) + DOT, t)
    t = _DECIMAL.sub(DOT, t)
    parts = [p.replace(DOT, ".").strip() for p in _SPLIT.split(t) if p.strip()]
    out = []
    for p in parts:   # very long "sentences" (quoted statutes) are split on semicolons
        if len(p.split()) > max_words:
            out.extend(s.strip() for s in re.split(r"(?<=;)\s+", p) if s.strip())
        else:
            out.append(p)
    merged = []
    for s in out:     # glue fragments like "(supra)." onto the previous sentence
        if merged and len(s.split()) < min_words:
            merged[-1] = merged[-1] + " " + s
        else:
            merged.append(s)
    return merged


STATUTE_LINE = re.compile(r"^\W*(\d{3}[A-Z]?\.\s+[A-Z]|(First|Second|Third|Fourth|Fifth|Sixth|Seventh|"
                          r"Eighth|Eight|Ninth|Tenth)\s+Exception\b|Explanation\s*\d*\s*[.:-])")
PARTY_ARG = re.compile(
    r"^\W*(the\s+)?(learned\s+)?(senior\s+)?(counsel|advocate|petitioner|complainant|respondent|appellant|accused)"
    r"s?\b.{0,80}?\b(submit|submits|submitted|submissions?|contend|contends|contended|contentions?|argue|argues|argued|"
    r"arguments?|urged|vehemently|claims?|claimed|places? reliance|relies|relied)\b"
    r"|^\W*it\s+(is|was|has\s+been)\s+(submitted|contended|argued|urged)", re.I)

INTRO_QUOTE = re.compile(r"\b(as under|as follows|reads? thus|provides? (as|that))\s*:\s*[\d.\s]*$", re.I)

def quote_flags(sents):
    flags, open_ = [], False
    for s in sents:
        flags.append(open_ or s.lstrip().startswith(('"', "“")) or bool(STATUTE_LINE.match(s)))
        n_open, n_close = s.count("“"), s.count("”")
        if n_open > n_close:
            open_ = True
        elif n_close > n_open:
            open_ = False
        if s.count('"') % 2 == 1:
            open_ = not open_
        if INTRO_QUOTE.search(s):     # "read as under: 499." introduces unquoted statute text
            open_ = True
    return flags


def segment_doc(doc_id, text):
    rows = []
    paras, mode = split_numbered_paragraphs(clean_text(text))
    for p_idx, (p_no, p_text) in enumerate(paras):
        sents = split_sentences(p_text)
        for s_idx, (s, q) in enumerate(zip(sents, quote_flags(sents))):
            rows.append({"doc_id": doc_id, "para_idx": p_idx, "para_no": p_no, "sent_idx": s_idx,
                         "text": s, "in_quote": q, "party_argument": bool(PARTY_ARG.match(s)),
                         "seg_mode": mode})
    return rows


rows = []
for rec in corpus.itertuples(index=False):
    for field in TEXT_FIELDS:
        t = getattr(rec, field, None)
        if pd.isna(t) or not str(t).strip():
            continue
        for r in segment_doc(rec.doc_id, str(t)):
            r["section"] = field
            rows.append(r)
sents = pd.DataFrame(rows)
sents["n_words"] = sents.text.str.split().str.len()
sents = sents.reset_index(drop=True)

per_doc = sents.groupby("doc_id").agg(mode=("seg_mode", "first"),
                                      paragraphs=("para_idx", "nunique"),
                                      sentences=("text", "size"))
print(f"{len(sents):,} sentences from {sents.doc_id.nunique()} judgments")
print("\nSentences by section:\n", sents.section.value_counts().to_string())
print("\nSegmentation mode per judgment:\n", per_doc["mode"].value_counts().to_string())
print("\nParagraphs per judgment:\n", per_doc.paragraphs.describe().round(1).to_string())
print("\nSentences per judgment:\n", per_doc.sentences.describe().round(1).to_string())
print(f"\nQuoted sentences: {sents.in_quote.mean():.1%}   Party argument sentences: {sents.party_argument.mean():.1%}")

9,672 sentences from 297 judgments

Sentences by section:
 section
court_reasoning    7282
conclusion         2390

Segmentation mode per judgment:
 mode
none          276
numbered       17
blank_line      4

Paragraphs per judgment:
 count    297.0
mean       1.6
std        2.5
min        1.0
25%        1.0
50%        1.0
75%        1.0
max       25.0

Sentences per judgment:
 count    297.0
mean      32.6
std       48.3
min        1.0
25%       10.0
50%       20.0
75%       34.0
max      525.0

Quoted sentences: 21.2%   Party argument sentences: 0.8%


In [4]:
# Spot check: one judgment's first sentences. Read these before trusting anything below.
example = per_doc.sort_values("paragraphs").index[len(per_doc) // 2]
print(TITLE.get(example, example))
sents.loc[sents.doc_id == example, ["para_no", "sent_idx", "in_quote", "party_argument", "text"]].head(15)

Smt.Mauna M.H. W/O K.M. Shivaprasad vs K.G. Mahesh Murthy on 26 November, 2020


,para_no,sent_idx,in_quote,party_argument,text
2696,NaN,0,False,False,I have perused the entire materials placed before this Court.
2697,NaN,1,False,True,Learned counsel for the respondent in his argument in the opening sentence stated that there was no reason for the petitioner to produce Ex.P.68.
2698,NaN,2,False,False,"It may be so according to him, but the petitioner might have reasons for producing several of the documents, as such, she has produced 202 documents in the ..."
2699,NaN,3,False,False,"It was for her to prove necessity for producing those documents, if she feels them to be necessary, otherwise those documents go without carrying any value ..."
2700,NaN,4,False,False,"As already observed, even in her examination-in-chief also except mentioning that she is producing a bill and marking it as Ex.P.68 she had not even whisper..."
2701,NaN,5,False,False,In such a scenario nothing had warranted the respondent therein to put specific question to PW1 in her cross-examination about Ex.P.68 and elicit reply from...
2702,NaN,6,False,False,"It is only since certain questions were put to her, she has given the answer as extracted above."
2703,NaN,7,False,False,The first question was in the form of statement suggesting to witness that said Ex.P.68 was with respect to purchasing of liquor from Military Canteen.
2704,NaN,8,False,False,The said statement was admitted as true by the witnesses.
2705,NaN,9,False,False,"Thus, the very first suggestion that the said document is with respect to purchase of liquor was made by none else than the respondent in the said case, who..."


## Save

In [5]:
sents.to_pickle(SENTS_PKL)
print(f"Saved {len(sents):,} sentences to {SENTS_PKL.relative_to(BASE)}")

Saved 9,672 sentences to argument_mining/intermediate/sentences.pkl
